# FF3: Welche Lebensstil-Bündel tragen die politische Sortierung?

In [1]:
source("setup.R")

CTFIDF_CSV <- file.path(PROJ, "Data/ctfidf_top50_LABEL2016.csv")
ZERLEGUNG  <- file.path(OUT_DIR, "ff2_zerlegung_gewichtet.csv")

data.table 1.17.8 using 8 threads (see ?getDTthreads).  Latest news: r-datatable.com

Attaching package: ‘igraph’

The following objects are masked from ‘package:stats’:

    decompose, spectrum

The following object is masked from ‘package:base’:

    union


Attaching package: ‘dbscan’

The following object is masked from ‘package:stats’:

    as.dendrogram

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ lubridate::%--%()      masks igraph::%--%()
✖ dplyr::as_data_frame() masks tibble::as_data_frame(), igraph::as_data_frame()
✖ dplyr::between()       masks data.table::between()
✖ purrr::compose()       masks igraph::compose()
✖ tidyr::crossing()      masks igraph::crossing()
✖ dplyr::filte

Warning messages:
1: package ‘igraph’ was built under R version 4.5.3 
2: package ‘ineq’ was built under R version 4.5.2 


## Grundlage

In [2]:
top50 <- read_csv(CTFIDF_CSV,   show_col_types = FALSE)
part  <- read_csv(CLUSTER_CSV,  show_col_types = FALSE)

fix2016  <- part |> filter(jahr == 2016, cluster != -1) |> select(subreddit, cluster)
groessen <- fix2016 |> count(cluster, name = "n_subs")

scores <- read_csv(SCORE_CSV, show_col_types = FALSE) |>
  rename(subreddit = 1) |>
  pivot_longer(starts_with("score_"), names_to = "jahr", values_to = "score",
               names_prefix = "score_", names_transform = list(jahr = as.integer))

ref <- scores |> filter(jahr == 2016, !is.na(score))
mu0 <- mean(ref$score); sd0 <- sd(ref$score)
cat("z-Referenz 2016 (volles Embedding): n =", nrow(ref),
    " mu =", round(mu0, 6), " sd =", round(sd0, 6), "\n")

zr <- scores |>
  filter(!is.na(score)) |>
  group_by(jahr) |>
  mutate(z   = (score - mu0) / sd0,
         pct = percent_rank(score),      # 0 bis 1, Position im Jahresfeld
         ext = abs(pct - 0.5) * 2) |>    # 0 = Mitte, 1 = Rand
  ungroup()

dat <- fix2016 |> inner_join(zr, by = "subreddit")

k_ctfidf <- top50 |> filter(year == 2016) |> distinct(cluster) |> pull(cluster) |> sort()
cat("Cluster in c-TF-IDF:", length(k_ctfidf),
    "| in der fixen Partition:", n_distinct(fix2016$cluster), "\n")
cat("Subreddits in fix2016:", nrow(fix2016),
    "| Beobachtungen Sub x Jahr:", nrow(dat),
    "| davon 2016:", sum(dat$jahr == 2016), "\n")

# Ohne diesen Blick weiss niemand, welche Seite der Achse welche ist
anker <- c("The_Donald", "Conservative", "politics", "SandersForPresident", "hillaryclinton")
zr |> filter(jahr == 2016, subreddit %in% anker) |>
  select(subreddit, z, pct) |> arrange(z) |> print()

New names:
• `` -> `...1`
z-Referenz 2016 (volles Embedding): n = 16618  mu = 0.005044  sd = 0.067478 
Cluster in c-TF-IDF: 95 | in der fixen Partition: 95 
Subreddits in fix2016: 10941 | Beobachtungen Sub x Jahr: 96869 | davon 2016: 10941 
# A tibble: 5 × 3
  subreddit                z      pct
  <chr>                <dbl>    <dbl>
1 hillaryclinton      -5.20  0.000181
2 SandersForPresident -4.02  0.000782
3 politics            -0.772 0.207   
4 The_Donald           5.30  1.000   
5 Conservative         6.14  1       


In [3]:
fueller <- c("get","like","would","one","people","really","also","much","think","know",
             "good","even","still","well","time","make","want","way","things","see",
             "could","thanks","thank","please","new","use","using","need","first","back",
             "shit","fuck","lol","amp","yes","love","got","going","said","say","dont")

woerter <- function(cl, jahr = 2016, n = 12) {
  top50 |>
    filter(cluster == cl, year == jahr, !token %in% fueller) |>
    arrange(rang) |> slice_head(n = n) |> pull(token) |> paste(collapse = ", ")
}

zeig <- function(cl, jahr = 2016, n_mitglieder = 12) {
  cat("--- Cluster", cl, " n =", groessen$n_subs[groessen$cluster == cl], "Subreddits ---\n")
  cat("Woerter", jahr, ":", woerter(cl, jahr), "\n")
  if (jahr != 2016) cat("Woerter 2016 :", woerter(cl, 2016), "\n")
  m <- fix2016 |> filter(cluster == cl) |> pull(subreddit)
  cat("Mitglieder  :", paste(head(m, n_mitglieder), collapse = ", "),
      if (length(m) > n_mitglieder) " ..." else "", "\n\n")
}

uebersicht <- function(cluster_ids = NULL, jahr = 2016, n_woerter = 10) {
  d <- groessen
  if (!is.null(cluster_ids)) d <- d |> filter(cluster %in% cluster_ids)
  d |> arrange(desc(n_subs)) |>
    mutate(woerter = map_chr(cluster, woerter, jahr = jahr, n = n_woerter))
}

cat("Die 15 groessten Cluster 2016:\n")
print(uebersicht() |> slice_head(n = 15), n = 15, width = 200)

Die 15 groessten Cluster 2016:
# A tibble: 15 × 3
   cluster n_subs
     <dbl>  <int>
 1       3   1662
 2      92   1150
 3      22    660
 4      63    320
 5      94    282
 6      12    281
 7      15    240
 8      26    218
 9      75    191
10      36    183
11      23    170
12       6    154
13      88    154
14      58    148
15       1    139
   woerter                                                                      
   <chr>                                                                        
 1 sex, girl, sexy, looking, kik, cock, hot, cum, body, women                   
 2 game, hero, dota, bible, games, team, true, play, right, heroes              
 3 song, music, album, songs, guitar, band, sound, bass, track, rock            
 4 irl, guy, someone, right, actually, fucking, thing, something, never, man    
 5 team, year, game, season, teams, league, last, play, player, players         
 6 car, cars, engine, miles, drive, tires, driving, truck, oil, wheels       

## Bewegung und Beitrag der Bündel

In [4]:
reihen <- dat |>
  group_by(cluster, jahr) |>
  summarise(n_score = n(), mean_z = mean(z), mean_pct = mean(pct), mean_ext = mean(ext),
            .groups = "drop")

tau_von <- function(j, w) if (length(unique(w)) < 2) NA_real_ else cor(j, w, method = "kendall")

bewegung <- reihen |>
  group_by(cluster) |>
  summarise(n_2016   = n_score[jahr == 2016],
            n_2024   = n_score[jahr == 2024],
            pct_2016 = mean_pct[jahr == 2016],
            pct_2024 = mean_pct[jahr == 2024],
            d_pct    = pct_2024 - pct_2016,
            tau_pct  = tau_von(jahr, mean_pct),
            ext_2016 = mean_ext[jahr == 2016],
            ext_2024 = mean_ext[jahr == 2024],
            d_ext    = ext_2024 - ext_2016,
            tau_ext  = tau_von(jahr, mean_ext),
            z_2016   = mean_z[jahr == 2016],
            z_2024   = mean_z[jahr == 2024],
            d_z      = z_2024 - z_2016,
            .groups  = "drop") |>
  left_join(groessen, by = "cluster")

cat("Cluster:", nrow(bewegung), "| Mitglieder mit Score 2016:", sum(bewegung$n_2016),
    "| 2024:", sum(bewegung$n_2024),
    sprintf("(%.1f %% Abgang)\n", 100 * (1 - sum(bewegung$n_2024) / sum(bewegung$n_2016))))
cat("Rangkorrelation d_pct gegen d_z über die Cluster:",
    round(cor(bewegung$d_pct, bewegung$d_z, method = "spearman"), 3), "\n")

Cluster: 95 | Mitglieder mit Score 2016: 10941 | 2024: 10476 (4.3 % Abgang)
Rangkorrelation d_pct gegen d_z über die Cluster: 0.979 


In [5]:
beitrag <- dat |>
  group_by(jahr) |>
  mutate(gm = mean(z), N = n()) |>
  group_by(jahr, cluster) |>
  summarise(n_c = n(), N = first(N), gm = first(gm),
            mittel = mean(z), s2_c = mean((z - mean(z))^2), .groups = "drop") |>
  mutate(w    = n_c / N,
         d    = mittel - gm,
         b    = w * d^2,       # Beitrag zu between
         wthn = w * s2_c)      # Beitrag zu within

jahres <- beitrag |>
  group_by(jahr) |>
  summarise(N = first(N), gesamtmittel = first(gm),
            between_ff3 = sum(b), within_ff3 = sum(wthn), .groups = "drop") |>
  mutate(eta2_ff3 = between_ff3 / (between_ff3 + within_ff3))

zerl <- read_csv(ZERLEGUNG, show_col_types = FALSE)
pruef <- jahres |>
  inner_join(zerl |> select(jahr, N_ff2 = N, between_gew, within_gew, eta2), by = "jahr") |>
  mutate(diff_between = between_ff3 - between_gew,
         diff_within  = within_ff3  - within_gew,
         diff_eta2    = eta2_ff3    - eta2)

cat("max |Abweichung| between:", format(max(abs(pruef$diff_between)), scientific = TRUE),
    "| eta2:", format(max(abs(pruef$diff_eta2)), scientific = TRUE), "\n\n")

# Wohin bewegt sich die Feldmitte selbst? Ein Cluster kann Abstand gewinnen,
# ohne sich zu rühren, wenn das Gesamtmittel von ihm wegwandert.
print(jahres |> select(jahr, N, gesamtmittel, between_ff3, within_ff3, eta2_ff3), n = 20)

max |Abweichung| between: 6.161738e-15 | eta2: 5.245804e-15 

# A tibble: 9 × 6
   jahr     N gesamtmittel between_ff3 within_ff3 eta2_ff3
  <int> <int>        <dbl>       <dbl>      <dbl>    <dbl>
1  2016 10941      0.0418        0.136      0.732    0.157
2  2017 10850     -0.0211        0.170      0.746    0.185
3  2018 10842     -0.0378        0.174      0.801    0.178
4  2019 10820      0.0470        0.192      0.776    0.198
5  2020 10791      0.0382        0.181      0.759    0.192
6  2021 10780      0.00259       0.200      0.759    0.209
7  2022 10754     -0.0105        0.208      0.784    0.209
8  2023 10615      0.0416        0.175      0.757    0.188
9  2024 10476     -0.0327        0.215      0.768    0.219


In [6]:
b_wide <- beitrag |>
  filter(jahr %in% c(2016, 2024)) |>
  select(cluster, jahr, n_c, w, mittel, d, b) |>
  pivot_wider(names_from = jahr, values_from = c(n_c, w, mittel, d, b))

bt16    <- jahres$between_ff3[jahres$jahr == 2016]
bt24    <- jahres$between_ff3[jahres$jahr == 2024]
anstieg <- bt24 - bt16

zerlegt <- b_wide |>
  mutate(d_beitrag    = b_2024 - b_2016,
         w_quer       = (w_2016 + w_2024) / 2,
         d2_quer      = (d_2016^2 + d_2024^2) / 2,
         bei_position = w_quer * (d_2024^2 - d_2016^2),
         bei_anteil   = (w_2024 - w_2016) * d2_quer,
         rest         = d_beitrag - bei_position - bei_anteil,
         anteil_16_pz = 100 * b_2016 / bt16,
         anteil_24_pz = 100 * b_2024 / bt24,
         treiber      = if_else(abs(bei_position) >= abs(bei_anteil), "Position", "Anteil")) |>
  left_join(groessen, by = "cluster") |>
  mutate(woerter = map_chr(cluster, woerter, n = 8))

cat("between 2016:", round(bt16, 5), " 2024:", round(bt24, 5),
    " Anstieg:", round(anstieg, 5), "\n")
cat("davon über Position:", round(sum(zerlegt$bei_position), 5),
    sprintf("(%+.1f %%)\n", 100 * sum(zerlegt$bei_position) / anstieg))
cat("davon über Anteil  :", round(sum(zerlegt$bei_anteil), 5),
    sprintf("(%+.1f %%)\n", 100 * sum(zerlegt$bei_anteil) / anstieg))

between 2016: 0.13609  2024: 0.21536  Anstieg: 0.07927 
davon über Position: 0.07544 (+95.2 %)
davon über Anteil  : 0.00383 (+4.8 %)


In [7]:
sort_d   <- sort(zerlegt$d_beitrag, decreasing = TRUE)
sort_abs <- sort(abs(zerlegt$d_beitrag), decreasing = TRUE)

for (k in c(5, 10, 20)) {
  cat(sprintf("  Top %2d: %6.1f %% des Netto-Anstiegs | %5.1f %% der gesamten Bewegung\n",
              k, 100 * sum(head(sort_d, k)) / anstieg,
              100 * sum(head(sort_abs, k)) / sum(sort_abs)))
}
cat(sprintf("  Cluster mit sinkendem Beitrag: %d von %d\n",
            sum(zerlegt$d_beitrag < 0), nrow(zerlegt)))
cat("  Bei Gleichverteilung wären es",
    sprintf("%.1f / %.1f / %.1f %%\n", 500 / nrow(zerlegt), 1000 / nrow(zerlegt),
            2000 / nrow(zerlegt)))

spalten <- c("cluster", "n_subs", "n_c_2016", "n_c_2024", "mittel_2016", "mittel_2024",
             "anteil_16_pz", "anteil_24_pz", "d_beitrag", "bei_position", "bei_anteil",
             "treiber", "woerter")

cat("\n=== Groesster Zuwachs am Sortierungsbeitrag ===\n")
print(zerlegt |> arrange(desc(d_beitrag)) |> slice_head(n = 15) |> select(all_of(spalten)),
      n = 15, width = 230)

cat("\n=== Groesster Rueckgang, also politisch unauffaelliger geworden ===\n")
print(zerlegt |> arrange(d_beitrag) |> slice_head(n = 10) |> select(all_of(spalten)),
      n = 10, width = 230)

# Hier ist der Mittelwert stehengeblieben und das Cluster nur größer geworden.
cat("\n=== Zuwachs ueberwiegend aus Groesse statt Position ===\n")
print(zerlegt |> filter(treiber == "Anteil", d_beitrag > 0) |> arrange(desc(d_beitrag)) |>
        slice_head(n = 10) |> select(all_of(spalten)), n = 10, width = 230)

  Top  5:   67.2 % des Netto-Anstiegs |  37.7 % der gesamten Bewegung
  Top 10:   87.0 % des Netto-Anstiegs |  52.4 % der gesamten Bewegung
  Top 20:  113.1 % des Netto-Anstiegs |  68.9 % der gesamten Bewegung
  Cluster mit sinkendem Beitrag: 34 von 95
  Bei Gleichverteilung wären es 5.3 / 10.5 / 21.1 %

=== Groesster Zuwachs am Sortierungsbeitrag ===
# A tibble: 15 × 13
   cluster n_subs n_c_2016 n_c_2024 mittel_2016 mittel_2024 anteil_16_pz
     <dbl>  <int>    <int>    <int>       <dbl>       <dbl>        <dbl>
 1      26    218      218      216     0.516         1.08       3.30   
 2      17     95       95       95     0.826         1.28       3.92   
 3       3   1662     1662     1438     0.00282      -0.288      0.169  
 4      88    154      154      151     0.0246        0.720      0.00305
 5       8     62       62       62     0.724         1.11       1.94   
 6      87     66       66       66     0.793         0.993      2.50   
 7      14     85       85       85    -0.

## Gegenproben

In [8]:
decke <- bewegung |> inner_join(select(zerlegt, cluster, d_2016, bei_position), by = "cluster")
korr <- c(
  pct16_gegen_dpct    = cor(decke$pct_2016,    decke$d_pct,        method = "spearman"),
  ext16_gegen_dext    = cor(decke$ext_2016,    decke$d_ext,        method = "spearman"),
  abstand16_gegen_pos = cor(abs(decke$d_2016), decke$bei_position, method = "spearman"))
cat("Deckeneffekt, Spearman (Ausgangslage 2016 gegen spaetere Bewegung):\n")
print(round(korr, 3))

# Seitenzuordnung nach der Lage 2016. Wer die Seite wechselt, bleibt bei seiner
# Ausgangsseite, damit die Gruppen über die Zeit fest sind.
seiten <- zerlegt |>
  mutate(seite = if_else(d_2016 > 0, "positiv", "negativ")) |>
  group_by(seite) |>
  summarise(n_cluster = n(), subs_2016 = sum(n_c_2016), subs_2024 = sum(n_c_2024),
            beitrag_2016 = sum(b_2016), beitrag_2024 = sum(b_2024),
            zuwachs = sum(d_beitrag), via_position = sum(bei_position),
            via_anteil = sum(bei_anteil), .groups = "drop") |>
  mutate(anteil_am_zuwachs_pz = 100 * zuwachs / sum(zuwachs))
cat("\nAsymmetrie der Seiten:\n")
print(seiten, width = 200)

cat("\nRichtungsbewegung, gewichtet mit der Clustergroesse 2016:\n")
cat("  mean d_pct:", round(weighted.mean(bewegung$d_pct, bewegung$n_2016), 4),
    "| mean d_ext:", round(weighted.mean(bewegung$d_ext, bewegung$n_2016), 4),
    "| nach oben:", sum(bewegung$d_pct > 0), "| nach unten:", sum(bewegung$d_pct < 0), "\n")

# Getrennt wird am Mittelwert des Feldes, nicht am Nullpunkt der Skala. Das
# Feldmittel liegt 2016 nicht bei null, Cluster mit leicht positivem Score
# zählen deshalb nach links. Zwei der größten Beiträge liegen in diesem
# Streifen, an ihnen hängt fast die ganze Zahl. Beide Varianten stehen hier,
# berichtet wird die konservative, weil die übrige Auswertung durchgehend
# gegen das Feldmittel rechnet.
feldmittel <- mean(zerlegt$mittel_2016 - zerlegt$d_2016)
cat("\nFeldmittel 2016 auf der standardisierten Skala:", round(feldmittel, 4), "\n")

seiten_regel <- function(regel) {
  zerlegt |>
    mutate(seite = if_else(regel, "positiv", "negativ")) |>
    group_by(seite) |>
    summarise(n_cluster = n(), subs = sum(n_c_2016), zuwachs = sum(d_beitrag),
              .groups = "drop") |>
    mutate(anteil_pz = 100 * zuwachs / sum(zuwachs))
}
cat("\nTrennung am Feldmittel (berichtet):\n");           print(seiten_regel(zerlegt$d_2016 > 0))
cat("\nTrennung am Nullpunkt der Skala (Gegenprobe):\n"); print(seiten_regel(zerlegt$mittel_2016 > 0))

grenz <- zerlegt |>
  filter(mittel_2016 > 0, d_2016 < 0) |>
  arrange(desc(d_beitrag)) |>
  transmute(cluster, n_c_2016, mittel_2016 = round(mittel_2016, 4),
            anteil_pz = round(100 * d_beitrag / sum(zerlegt$d_beitrag), 1))
cat("\nCluster im Streifen zwischen Nullpunkt und Feldmittel:\n")
print(grenz)
cat("Summe ihres Beitrags:", round(sum(grenz$anteil_pz), 1), "Prozent des Anstiegs.\n")

Deckeneffekt, Spearman (Ausgangslage 2016 gegen spaetere Bewegung):
   pct16_gegen_dpct    ext16_gegen_dext abstand16_gegen_pos 
             -0.136              -0.248              -0.245 

Asymmetrie der Seiten:
# A tibble: 2 × 10
  seite   n_cluster subs_2016 subs_2024 beitrag_2016 beitrag_2024 zuwachs
  <chr>       <int>     <int>     <int>        <dbl>        <dbl>   <dbl>
1 negativ        50      6227      5923       0.0687       0.0923  0.0236
2 positiv        45      4714      4553       0.0673       0.123   0.0557
  via_position via_anteil anteil_am_zuwachs_pz
         <dbl>      <dbl>                <dbl>
1       0.0224    0.00121                 29.8
2       0.0531    0.00262                 70.2

Richtungsbewegung, gewichtet mit der Clustergroesse 2016:
  mean d_pct: 2e-04 | mean d_ext: -0.0032 | nach oben: 52 | nach unten: 43 

Feldmittel 2016 auf der standardisierten Skala: 0.0418 

Trennung am Feldmittel (berichtet):
# A tibble: 2 × 5
  seite   n_cluster  subs zuwachs an

In [9]:
# Cluster mit Ortsbezug, an den kennzeichnenden Woertern und an der
# Mitgliedschaft bestimmt. Nicht aufgenommen: 31 (Star Wars, das Wort "luke"
# enthält zufällig "uk") und 47 (Karten und Flaggen, kein Ortscluster).
GEO_ALLE <- c(4,   # sydney, melbourne, afl, australia, brisbane, nrl
              6,   # vancouver, canada, city, pay, tax
              8,   # austin, houston, texas, romo, dak
              14,  # cricket, india, bowling, kohli, dubai, pakistan
              17,  # london, pay, work, tax
              34,  # nyc, manhattan, brooklyn, subway, rent
              45,  # seattle, mariners, seahawks, portland
              56)  # san, giants, lakers, dodgers
GEO_NONUS <- c(4, 6, 14, 17)

cat(sprintf("Subreddits 2016 in den Ortsclustern: %d von %d (%.1f %%), davon ausserhalb der USA %d (%.1f %%)
",
            sum(fix2016$cluster %in% GEO_ALLE), nrow(fix2016),
            100 * mean(fix2016$cluster %in% GEO_ALLE),
            sum(fix2016$cluster %in% GEO_NONUS),
            100 * mean(fix2016$cluster %in% GEO_NONUS)))

# Beim Ausschluss werden Feldmitte und Gewichte neu bestimmt, deshalb die volle
# Zerlegung je Variante und nicht ein Filter auf die fertige Tabelle.
zerlegung_ohne <- function(drop = integer(0)) {
  dat |> filter(!cluster %in% drop, jahr %in% c(2016, 2024)) |>
    group_by(jahr) |> mutate(gm = mean(z), N = n()) |>
    group_by(jahr, cluster) |>
    summarise(n_c = n(), w = n() / first(N), mittel = mean(z),
              gm = first(gm), .groups = "drop") |>
    mutate(d = mittel - gm, b = w * d^2) |>
    select(jahr, cluster, n_c, w, d, b, mittel) |>
    pivot_wider(names_from = jahr, values_from = c(n_c, w, d, b, mittel)) |>
    mutate(d_beitrag = b_2024 - b_2016)
}

asym <- function(drop = integer(0), regel = "feldmitte") {
  zl <- zerlegung_ohne(drop) |>
    mutate(rechts = if (regel == "feldmitte") d_2016 > 0 else mittel_2016 > 0)
  tibble(n_cluster = nrow(zl), n_rechts = sum(zl$rechts),
         subs_pz = 100 * sum(zl$n_c_2016[zl$rechts]) / sum(zl$n_c_2016),
         zuwachs_pz = 100 * sum(zl$d_beitrag[zl$rechts]) / sum(zl$d_beitrag))
}

gegen_seiten <- bind_rows(
  asym()                    |> mutate(variante = "alle 95, Feldmitte"),
  asym(regel = "nullpunkt") |> mutate(variante = "alle 95, Nullpunkt"),
  asym(GEO_ALLE)            |> mutate(variante = "ohne die Ortscluster"),
  asym(GEO_NONUS)           |> mutate(variante = "ohne die ausserhalb der USA")) |>
  select(variante, n_cluster, n_rechts, subs_pz, zuwachs_pz) |>
  mutate(across(where(is.numeric), \(x) round(x, 1)))

print(as.data.frame(gegen_seiten))
write_csv(gegen_seiten, file.path(OUT_DIR, "ff3_seiten_gegenproben.csv"))


Subreddits 2016 in den Ortsclustern: 610 von 10941 (5.6 %), davon ausserhalb der USA 408 (3.7 %)
                     variante n_cluster n_rechts subs_pz zuwachs_pz
1          alle 95, Feldmitte        95       45    43.1       70.2
2          alle 95, Nullpunkt        95       51    64.2       92.6
3        ohne die Ortscluster        87       42    43.4       66.9
4 ohne die ausserhalb der USA        91       44    43.5       70.6


In [10]:
# Wie viel des eta2-Anstiegs bleibt ohne die Ortscluster stehen? Auch hier wird
# die Feldmitte je Variante neu bestimmt.
reihe_ohne <- function(drop = integer(0)) {
  d  <- dat |> filter(!cluster %in% drop)
  gm <- d |> group_by(jahr) |> summarise(gm = mean(z), .groups = "drop")
  d |> left_join(gm, by = "jahr") |>
    group_by(jahr, cluster) |>
    summarise(n_c = n(), mittel = mean(z), s2 = mean((z - mean(z))^2),
              gm = first(gm), .groups = "drop") |>
    group_by(jahr) |> mutate(N = sum(n_c), w = n_c / N) |>
    summarise(N = first(N), k = n(),
              between = sum(w * (mittel - gm)^2),
              within = sum(w * s2), .groups = "drop") |>
    mutate(eta2 = between / (between + within))
}

kennz <- function(drop = integer(0)) {
  r <- reihe_ohne(drop)
  tibble(k_2016 = r$k[1], N_2016 = r$N[1],
         eta2_2016 = r$eta2[1], eta2_2024 = r$eta2[nrow(r)],
         d_eta2 = r$eta2[nrow(r)] - r$eta2[1],
         tau = cor(r$jahr, r$eta2, method = "kendall"))
}

basis <- kennz()
geo_erhalt <- bind_rows(
  basis             |> mutate(variante = "alle 95 Cluster"),
  kennz(GEO_ALLE)   |> mutate(variante = "ohne die Ortscluster"),
  kennz(GEO_NONUS)  |> mutate(variante = "ohne die ausserhalb der USA")) |>
  mutate(rest_pz = 100 * d_eta2 / basis$d_eta2) |>
  select(variante, k_2016, N_2016, eta2_2016, eta2_2024, d_eta2, rest_pz, tau) |>
  mutate(across(where(is.numeric), \(x) round(x, 4)))

print(as.data.frame(geo_erhalt))
write_csv(geo_erhalt, file.path(OUT_DIR, "ff3_geo_erhalt.csv"))


                     variante k_2016 N_2016 eta2_2016 eta2_2024 d_eta2  rest_pz
1             alle 95 Cluster     95  10941    0.1569    0.2190 0.0621 100.0000
2        ohne die Ortscluster     87  10331    0.1451    0.1947 0.0496  79.9102
3 ohne die ausserhalb der USA     91  10533    0.1557    0.2057 0.0501  80.6122
     tau
1 0.6667
2 0.5000
3 0.5000


In [11]:
# Nullmodell: liegt der Rückgang an diesen Clustern oder daran, dass überhaupt
# welche fehlen? Für jedes Ortscluster wird eines ähnlicher Größe gezogen,
# damit die Größenstruktur des Ausschlusses erhalten bleibt.
set.seed(42)
NSIM_GEO <- 999
d_voll   <- basis$d_eta2
obs      <- kennz(GEO_ALLE)
pool_all <- setdiff(sort(unique(dat$cluster)), GEO_ALLE)
gr       <- setNames(groessen$n_subs, groessen$cluster)

zieh_matched <- function() {
  pool <- pool_all; out <- integer(0)
  for (g in GEO_ALLE) {
    kand <- pool[order(abs(gr[as.character(pool)] - gr[as.character(g)]))][seq_len(min(10, length(pool)))]
    pick <- if (length(kand) == 1) kand else sample(kand, 1)
    out <- c(out, pick); pool <- setdiff(pool, pick)
  }
  out
}

sim_geo <- map_dfr(seq_len(NSIM_GEO), function(i) {
  drop <- zieh_matched(); k <- kennz(drop)
  tibble(n_subs = sum(fix2016$cluster %in% drop),
         rest_pz = 100 * k$d_eta2 / d_voll, tau = k$tau)
})

geo_null <- tibble(
  rest_beobachtet = 100 * obs$d_eta2 / d_voll,
  rest_median = median(sim_geo$rest_pz),
  rest_p = mean(sim_geo$rest_pz <= 100 * obs$d_eta2 / d_voll),
  tau_beobachtet = obs$tau, tau_median = median(sim_geo$tau),
  tau_p = mean(sim_geo$tau <= obs$tau),
  subs_beobachtet = sum(fix2016$cluster %in% GEO_ALLE),
  subs_median = median(sim_geo$n_subs), n_sim = NSIM_GEO) |>
  mutate(across(where(is.numeric), \(x) round(x, 3)))

print(as.data.frame(geo_null))
write_csv(geo_null, file.path(OUT_DIR, "ff3_geo_nullmodell.csv"))


  rest_beobachtet rest_median rest_p tau_beobachtet tau_median tau_p
1           79.91     101.014      0            0.5      0.611 0.032
  subs_beobachtet subs_median n_sim
1             610         596   999


In [12]:
vollstaendig <- dat |> count(subreddit) |> filter(n == length(JAHRE)) |> pull(subreddit)
cat("Subreddits mit Score in allen", length(JAHRE), "Jahren:", length(vollstaendig),
    "von", n_distinct(dat$subreddit), "\n")

bal <- dat |>
  filter(subreddit %in% vollstaendig) |>
  group_by(jahr) |> mutate(gm = mean(z), N = n()) |>
  group_by(cluster, jahr) |>
  summarise(w = n() / first(N), d = mean(z) - first(gm),
            mean_pct = mean(pct), mean_ext = mean(ext), .groups = "drop") |>
  mutate(b = w * d^2) |>
  group_by(cluster) |>
  summarise(d_beitrag_b = b[jahr == 2024] - b[jahr == 2016],
            d_pct_b     = mean_pct[jahr == 2024] - mean_pct[jahr == 2016],
            d_ext_b     = mean_ext[jahr == 2024] - mean_ext[jahr == 2016], .groups = "drop")

vgl_bal <- zerlegt |> select(cluster, n_subs, d_beitrag) |>
  inner_join(select(bewegung, cluster, d_pct, d_ext), by = "cluster") |>
  inner_join(bal, by = "cluster")
cat("Uebereinstimmung voll gegen balanciert (Spearman ueber die Cluster):\n")
cat("  d_beitrag:", round(cor(vgl_bal$d_beitrag, vgl_bal$d_beitrag_b, method = "spearman"), 3),
    "| d_pct:",     round(cor(vgl_bal$d_pct,     vgl_bal$d_pct_b,     method = "spearman"), 3),
    "| d_ext:",     round(cor(vgl_bal$d_ext,     vgl_bal$d_ext_b,     method = "spearman"), 3), "\n")
print(vgl_bal |> mutate(diff = abs(d_beitrag - d_beitrag_b)) |> arrange(desc(diff)) |>
        slice_head(n = 8) |> select(cluster, n_subs, d_beitrag, d_beitrag_b, diff), n = 8)

Subreddits mit Score in allen 9 Jahren: 10266 von 10941 
Uebereinstimmung voll gegen balanciert (Spearman ueber die Cluster):
  d_beitrag: 0.994 | d_pct: 0.995 | d_ext: 0.994 
# A tibble: 8 × 5
  cluster n_subs d_beitrag d_beitrag_b     diff
    <dbl>  <int>     <dbl>       <dbl>    <dbl>
1      78     40 -0.000805   -0.000191 0.000614
2       7     60  0.00128     0.000709 0.000574
3       3   1662  0.00874     0.00820  0.000539
4      60     52 -0.00753    -0.00802  0.000487
5      24     67 -0.00187    -0.00154  0.000325
6      92   1150 -0.00709    -0.00741  0.000322
7      41     30  0.00102     0.00132  0.000305
8      87     66  0.00323     0.00292  0.000300


In [13]:
set.seed(42)
NSIM <- 999L

paare <- dat |>
  filter(jahr %in% c(2016, 2024)) |> select(subreddit, jahr, pct) |>
  pivot_wider(names_from = jahr, values_from = pct, names_prefix = "pct_") |>
  filter(!is.na(pct_2016), !is.na(pct_2024))

# Etikettenvorrat in den echten Clustergrößen. Er ist länger als paare, weil
# nicht jeder Sub in beiden Jahren einen Score hat, deshalb wird gezogen.
etiketten <- rep(groessen$cluster, times = groessen$n_subs)

null_q90 <- replicate(NSIM, {
  paare |>
    mutate(pseudo = sample(etiketten, nrow(paare))) |>
    group_by(pseudo) |>
    summarise(d = mean(pct_2024) - mean(pct_2016), .groups = "drop") |>
    pull(d) |> abs() |> quantile(0.90)
})
beob_q90 <- quantile(abs(bewegung$d_pct), 0.90, na.rm = TRUE)

cat("90er-Perzentil von |d_pct| beobachtet:", round(beob_q90, 4),
    "| Nullmodell:", round(mean(null_q90), 4),
    sprintf("(95 %%-Bereich %.4f bis %.4f)\n",
            quantile(null_q90, 0.025), quantile(null_q90, 0.975)))

90er-Perzentil von |d_pct| beobachtet: 0.1625 | Nullmodell: 0.0678 (95 %-Bereich 0.0550 bis 0.0810)


In [14]:
ung <- zerlegt |>
  transmute(cluster,
            b_ung_2016 = d_2016^2,
            b_ung_2024 = d_2024^2,
            d_ung      = b_ung_2024 - b_ung_2016)

anstieg_ung <- sum(ung$d_ung)
top_ung  <- function(k) 100 * sum(sort(ung$d_ung, decreasing = TRUE)[1:k]) / anstieg_ung
topb_ung <- function(k) 100 * sum(sort(abs(ung$d_ung), decreasing = TRUE)[1:k]) / sum(abs(ung$d_ung))

cat(sprintf("Summe der ungewichteten Abstaende: %.4f (2016) -> %.4f (2024), Anstieg %.4f\n",
            sum(ung$b_ung_2016), sum(ung$b_ung_2024), anstieg_ung))
cat(sprintf("  netto   Top5 %5.1f %%  Top10 %5.1f %%  Top20 %5.1f %%\n",
            top_ung(5), top_ung(10), top_ung(20)))
cat(sprintf("  Betrag  Top5 %5.1f %%  Top10 %5.1f %%  Top20 %5.1f %%\n",
            topb_ung(5), topb_ung(10), topb_ung(20)))
cat(sprintf("  Cluster mit Rueckgang: %d von %d\n", sum(ung$d_ung < 0), nrow(ung)))

rang_ung <- zerlegt |> select(cluster, n_subs, d_beitrag) |>
  inner_join(select(ung, cluster, d_ung), by = "cluster")
cat(sprintf("  Rangkorrelation gewichtet gegen ungewichtet: %.3f\n",
            cor(rang_ung$d_beitrag, rang_ung$d_ung, method = "spearman")))
print(rang_ung |>
        mutate(rang_ung = rank(-d_ung), rang_gew = rank(-d_beitrag)) |>
        arrange(rang_ung) |> slice_head(n = 10) |>
        select(cluster, n_subs, d_ung, rang_ung, d_beitrag, rang_gew), n = 10)

Summe der ungewichteten Abstaende: 23.2746 (2016) -> 31.4165 (2024), Anstieg 8.1419
  netto   Top5  53.8 %  Top10  85.6 %  Top20 130.3 %
  Betrag  Top5  27.3 %  Top10  41.3 %  Top20  62.9 %
  Cluster mit Rueckgang: 34 von 95
  Rangkorrelation gewichtet gegen ungewichtet: 0.953
# A tibble: 10 × 6
   cluster n_subs d_ung rang_ung d_beitrag rang_gew
     <dbl>  <int> <dbl>    <dbl>     <dbl>    <dbl>
 1      17     95 1.10         1   0.0102         2
 2      26    218 1.01         2   0.0210         1
 3       8     62 0.833        3   0.00505        5
 4      42     38 0.795        4   0.00297       10
 5      41     30 0.636        5   0.00102       27
 6      88    154 0.567        6   0.00817        4
 7      53     56 0.527        7   0.00266       13
 8      77     31 0.501        8   0.00144       19
 9      31     34 0.495        9   0.00162       16
10      71     58 0.495       10   0.00252       14


## Kernzahlen sichern

In [15]:
beitrag_voll <- zerlegt |>
  left_join(bewegung |> select(cluster, pct_2016, pct_2024, d_pct, tau_pct,
                               ext_2016, ext_2024, d_ext, tau_ext), by = "cluster") |>
  arrange(desc(d_beitrag))

write_csv(beitrag_voll, file.path(OUT_DIR, "ff3_beitrag_zerlegung.csv"))
write_csv(jahres,       file.path(OUT_DIR, "ff3_beitrag_jahre.csv"))
write_csv(seiten,       file.path(OUT_DIR, "ff3_seiten.csv"))

kenn <- tibble(
  kennzahl = c("between_2016", "between_2024", "anstieg",
               "anteil_position_pz", "anteil_anteil_pz",
               "top5_netto_pz", "top10_netto_pz", "top20_netto_pz",
               "top5_betrag_pz", "top10_betrag_pz", "top20_betrag_pz",
               "gleichverteilung_top10_pz",
               "n_cluster", "n_cluster_rueckgang", "n_cluster_treiber_anteil",
               "decke_pct16_gegen_dpct", "decke_ext16_gegen_dext", "decke_abstand16_gegen_pos",
               "mean_d_pct_gew", "mean_d_ext_gew"),
  wert = c(bt16, bt24, anstieg,
           100 * sum(zerlegt$bei_position) / anstieg,
           100 * sum(zerlegt$bei_anteil)   / anstieg,
           100 * sum(head(sort_d, 5))  / anstieg,
           100 * sum(head(sort_d, 10)) / anstieg,
           100 * sum(head(sort_d, 20)) / anstieg,
           100 * sum(head(sort_abs, 5))  / sum(sort_abs),
           100 * sum(head(sort_abs, 10)) / sum(sort_abs),
           100 * sum(head(sort_abs, 20)) / sum(sort_abs),
           1000 / nrow(zerlegt),
           nrow(zerlegt), sum(zerlegt$d_beitrag < 0), sum(zerlegt$treiber == "Anteil"),
           korr[["pct16_gegen_dpct"]], korr[["ext16_gegen_dext"]], korr[["abstand16_gegen_pos"]],
           weighted.mean(bewegung$d_pct, bewegung$n_2016),
           weighted.mean(bewegung$d_ext, bewegung$n_2016)))

write_csv(kenn, file.path(OUT_DIR, "ff3_beitrag_kennzahlen.csv"))
print(kenn, n = 30)

# A tibble: 20 × 2
   kennzahl                        wert
   <chr>                          <dbl>
 1 between_2016                0.136   
 2 between_2024                0.215   
 3 anstieg                     0.0793  
 4 anteil_position_pz         95.2     
 5 anteil_anteil_pz            4.83    
 6 top5_netto_pz              67.2     
 7 top10_netto_pz             87.0     
 8 top20_netto_pz            113.      
 9 top5_betrag_pz             37.7     
10 top10_betrag_pz            52.4     
11 top20_betrag_pz            68.9     
12 gleichverteilung_top10_pz  10.5     
13 n_cluster                  95       
14 n_cluster_rueckgang        34       
15 n_cluster_treiber_anteil    1       
16 decke_pct16_gegen_dpct     -0.136   
17 decke_ext16_gegen_dext     -0.248   
18 decke_abstand16_gegen_pos  -0.245   
19 mean_d_pct_gew              0.000204
20 mean_d_ext_gew             -0.00320 


## Stammbaum über die Mitgliedschaft

In [16]:
nat <- part |>
  filter(cluster != -1) |>
  inner_join(zr, by = c("subreddit", "jahr")) |>
  mutate(nr_key = paste0(jahr, "_", cluster))

nat_beitrag <- nat |>
  group_by(jahr) |> mutate(gm = mean(z), N = n()) |>
  group_by(jahr, cluster, nr_key) |>
  summarise(n_c = n(), N = first(N), gm = first(gm), mittel = mean(z),
            s2_c = mean((z - mean(z))^2), .groups = "drop") |>
  mutate(w = n_c / N, d = mittel - gm, b = w * d^2, wthn = w * s2_c)

nat_jahre <- nat_beitrag |>
  group_by(jahr) |>
  summarise(N = first(N), k = n(), between = sum(b), within = sum(wthn), .groups = "drop") |>
  mutate(eta2 = between / (between + within))

print(nat_jahre, n = 20)

# A tibble: 9 × 6
   jahr     N     k between within  eta2
  <dbl> <int> <int>   <dbl>  <dbl> <dbl>
1  2016 10941    95   0.136  0.732 0.157
2  2017 12285    95   0.179  0.733 0.196
3  2018 12897   120   0.221  0.733 0.232
4  2019 15312   143   0.245  0.707 0.257
5  2020 16647   165   0.237  0.674 0.260
6  2021 18338   177   0.268  0.683 0.282
7  2022 19077   201   0.303  0.702 0.302
8  2023 19550   212   0.316  0.638 0.331
9  2024 19338   200   0.360  0.665 0.351


In [17]:
SPLIT_MIN <- 0.25   # ab diesem Anteil zaehlt ein Zweig als eigene Abspaltung

mitglieder <- nat |> select(jahr, cluster, subreddit)
gr_nat     <- mitglieder |> count(jahr, cluster, name = "n")

kanten <- mitglieder |>
  rename(jahr_von = jahr, cluster_alt = cluster) |>
  mutate(jahr_bis = jahr_von + 1) |>
  inner_join(mitglieder |> rename(jahr_bis = jahr, cluster_neu = cluster),
             by = c("subreddit", "jahr_bis")) |>
  count(jahr_von, jahr_bis, cluster_alt, cluster_neu, name = "schnitt") |>
  left_join(gr_nat |> rename(jahr_von = jahr, cluster_alt = cluster, n_alt = n),
            by = c("jahr_von", "cluster_alt")) |>
  left_join(gr_nat |> rename(jahr_bis = jahr, cluster_neu = cluster, n_neu = n),
            by = c("jahr_bis", "cluster_neu")) |>
  mutate(anteil_alt = schnitt / n_alt,
         anteil_neu = schnitt / n_neu,
         jaccard    = schnitt / (n_alt + n_neu - schnitt))

abfluss <- kanten |>
  group_by(jahr_von, cluster_alt) |>
  summarise(abgegeben = sum(schnitt), n_alt = first(n_alt), .groups = "drop") |>
  mutate(verbleib = abgegeben / n_alt)

cat("Kanten:", nrow(kanten), "ueber", n_distinct(kanten$jahr_von), "Jahresuebergaenge\n")

cat("\nVerbleibsquote je Jahr: welcher Anteil der Mitglieder ist im Folgejahr noch\n",
    "in der Population, der Rest ist weggefallen oder gilt als Rauschen.\n")
print(abfluss |> group_by(jahr_von) |>
        summarise(cluster = n(), median_verbleib = round(median(verbleib), 3),
                  q10 = round(quantile(verbleib, 0.10), 3), .groups = "drop"))

best <- kanten |>
  group_by(jahr_von, cluster_alt) |>
  slice_max(anteil_alt, n = 1, with_ties = FALSE) |>
  ungroup()

cat("\nBester Nachfolger je Cluster, Mediane je Jahrespaar:\n")
print(best |> group_by(jahr_von) |>
        summarise(cluster = n(),
                  anteil_alt = round(median(anteil_alt), 3),
                  jaccard    = round(median(jaccard), 3),
                  anteil_neu = round(median(anteil_neu), 3), .groups = "drop"))
cat("Faellt der Jaccard staerker als anteil_alt, ist das kein Zerfall der Cluster,\n",
    "sondern der Zustrom neuer Subreddits im Nenner. Deshalb wird ueber anteil_alt\n",
    "verfolgt.\n")

# Vorwärts: in wie viele Zweige zerfällt ein Cluster. Rückwärts: aus wie
# vielen Quellen speist es sich. Beide Zahlen begrenzen, wie weit die Ketten als
# Fortbestand einer Gemeinschaft gelesen werden dürfen.
zweige  <- kanten |> filter(anteil_alt >= SPLIT_MIN) |> count(jahr_von, cluster_alt, name = "zweige")
quellen <- kanten |> filter(anteil_neu >= SPLIT_MIN) |> count(jahr_bis, cluster_neu, name = "quellen")

cat("\nAufspaltung (anteil_alt >=", SPLIT_MIN, "):\n")
print(zweige |> group_by(jahr_von) |>
        summarise(cluster = n(), mit_aufspaltung = sum(zweige > 1),
                  pz = round(100 * mean(zweige > 1), 1), .groups = "drop"))
cat("\nVerschmelzung (anteil_neu >=", SPLIT_MIN, "):\n")
print(quellen |> group_by(jahr_bis) |>
        summarise(cluster = n(), aus_verschmelzung = sum(quellen > 1),
                  pz = round(100 * mean(quellen > 1), 1), .groups = "drop"))

write_csv(kanten, file.path(OUT_DIR, "ff3_stammbaum_kanten.csv"))

Kanten: 3751 ueber 8 Jahresuebergaenge

Verbleibsquote je Jahr: welcher Anteil der Mitglieder ist im Folgejahr noch
 in der Population, der Rest ist weggefallen oder gilt als Rauschen.
# A tibble: 8 × 4
  jahr_von cluster median_verbleib   q10
     <dbl>   <int>           <dbl> <dbl>
1     2016      95           0.909 0.637
2     2017      95           0.928 0.669
3     2018     119           0.932 0.792
4     2019     142           0.911 0.708
5     2020     165           0.938 0.735
6     2021     177           0.917 0.714
7     2022     201           0.912 0.714
8     2023     210           0.927 0.681

Bester Nachfolger je Cluster, Mediane je Jahrespaar:
# A tibble: 8 × 5
  jahr_von cluster anteil_alt jaccard anteil_neu
     <dbl>   <int>      <dbl>   <dbl>      <dbl>
1     2016      95      0.867   0.624      0.762
2     2017      95      0.892   0.66       0.786
3     2018     119      0.914   0.653      0.769
4     2019     142      0.883   0.716      0.82 
5     2020     165   

In [18]:
LINIE_MIN  <- 0.50   # unter diesem anteil_alt gilt die Kette als gerissen
START_JAHR <- min(JAHRE)

start_cluster <- sort(unique(nat$cluster[nat$jahr == START_JAHR]))

linien <- map_dfr(start_cluster, function(c0) {
  cur  <- c0
  rows <- tibble(linie = c0, jahr = START_JAHR, cluster = c0,
                 anteil_alt = NA_real_, anteil_neu = NA_real_, jaccard = NA_real_)
  for (v in START_JAHR:(max(JAHRE) - 1)) {
    e <- best |> filter(jahr_von == v, cluster_alt == cur)
    if (nrow(e) == 0) break
    cur  <- e$cluster_neu[1]
    rows <- bind_rows(rows, tibble(linie = c0, jahr = v + 1, cluster = cur,
                                   anteil_alt = e$anteil_alt[1],
                                   anteil_neu = e$anteil_neu[1],
                                   jaccard    = e$jaccard[1]))
  }
  rows
})

# cummin sorgt dafür, dass eine gerissene Linie nicht wieder auflebt
intakt_nach <- function(jahr, wert, schwelle) {
  cummin(if_else(jahr == START_JAHR, 1, as.numeric(wert >= schwelle))) == 1
}

linien <- linien |>
  group_by(linie) |> arrange(jahr, .by_group = TRUE) |>
  mutate(intakt    = intakt_nach(jahr, anteil_alt, LINIE_MIN),
         intakt_ja = intakt_nach(jahr, jaccard,    LINIE_MIN)) |>
  ungroup()

linien_z <- linien |>
  left_join(nat_beitrag |> select(jahr, cluster, n_c, mittel, w, d, b),
            by = c("jahr", "cluster"))

cat("Startcluster", START_JAHR, ":", length(start_cluster), "\n\n")
cat("Intakte Linien je Jahr, beide Kriterien (Schwelle", LINIE_MIN, "):\n")
print(linien |> group_by(jahr) |>
        summarise(nach_anteil_alt = sum(intakt), nach_jaccard = sum(intakt_ja),
                  .groups = "drop"))
cat("Die Differenz der beiden Spalten ist Populationswachstum und kein\n",
    "Zerfall der Cluster.\n")

write_csv(linien_z, file.path(OUT_DIR, "ff3_stammbaum_linien.csv"))

Startcluster 2016 : 95 

Intakte Linien je Jahr, beide Kriterien (Schwelle 0.5 ):
# A tibble: 9 × 3
   jahr nach_anteil_alt nach_jaccard
  <dbl>           <int>        <int>
1  2016              95           95
2  2017              85           66
3  2018              75           56
4  2019              70           50
5  2020              70           48
6  2021              69           47
7  2022              68           45
8  2023              60           43
9  2024              56           39
Die Differenz der beiden Spalten ist Populationswachstum und kein
 Zerfall der Cluster.


In [19]:
j0 <- min(JAHRE); j1 <- max(JAHRE)

linien_intakt <- linien_z |>
  filter(intakt, jahr <= j1) |>
  group_by(linie) |> filter(n() == length(j0:j1)) |> ungroup()

cat("Linien,", j0, "bis", j1, "durchgehend intakt:",
    n_distinct(linien_intakt$linie), "von", length(start_cluster), "\n")

lin_bew <- linien_intakt |>
  group_by(linie) |>
  summarise(cl_von = cluster[jahr == j0], cl_bis = cluster[jahr == j1],
            n_von  = n_c[jahr == j0],     n_bis  = n_c[jahr == j1],
            z_von  = mittel[jahr == j0],  z_bis  = mittel[jahr == j1],
            d_z    = z_bis - z_von,
            tau_z  = tau_von(jahr, mittel),
            w_von  = w[jahr == j0], w_bis = w[jahr == j1],
            d_von  = d[jahr == j0], d_bis = d[jahr == j1],
            b_von  = b[jahr == j0], b_bis = b[jahr == j1],
            min_anteil = min(anteil_alt, na.rm = TRUE),
            .groups = "drop") |>
  mutate(d_beitrag    = b_bis - b_von,
         w_quer       = (w_von + w_bis) / 2,
         d2_quer      = (d_von^2 + d_bis^2) / 2,
         bei_position = w_quer * (d_bis^2 - d_von^2),
         bei_anteil   = (w_bis - w_von) * d2_quer,
         rest         = d_beitrag - bei_position - bei_anteil,
         treiber      = if_else(abs(bei_position) >= abs(bei_anteil), "Position", "Anteil"))

lin_bew <- lin_bew |>
  mutate(woerter_von = map_chr(cl_von, woerter, jahr = j0, n = 7),
         woerter_bis = map_chr(cl_bis, woerter, jahr = j1, n = 7))

cat("Summe der Linien-Beiträge", j0, ":", round(sum(lin_bew$b_von), 5),
    "|", j1, ":", round(sum(lin_bew$b_bis), 5), "\n")
cat("Gesamtes natives between  ", j0, ":", round(nat_jahre$between[nat_jahre$jahr == j0], 5),
    "|", j1, ":", round(nat_jahre$between[nat_jahre$jahr == j1], 5), "\n")
cat("Die Luecke ist der Teil der Landschaft, den die ueberlebenden Linien nicht abdecken.\n")

sp <- c("linie", "cl_bis", "n_von", "n_bis", "z_von", "z_bis", "d_z", "tau_z",
        "d_beitrag", "bei_position", "bei_anteil", "treiber", "min_anteil", "woerter_bis")

cat("\n=== Groesste Bewegung in positive Richtung ===\n")
print(lin_bew |> arrange(desc(d_z)) |> slice_head(n = 12) |> select(all_of(sp)), n = 12, width = 240)
cat("\n=== Groesste Bewegung in negative Richtung ===\n")
print(lin_bew |> arrange(d_z) |> slice_head(n = 12) |> select(all_of(sp)), n = 12, width = 240)
cat("\n=== Groesster Zuwachs am Sortierungsbeitrag entlang der Linie ===\n")
print(lin_bew |> arrange(desc(d_beitrag)) |> slice_head(n = 12) |> select(all_of(sp)),
      n = 12, width = 240)

# Beide Wege starten bei denselben 2016er-Clustern, der eine friert die
# Mitgliedschaft ein, der andere lässt sie mitlaufen.
vgl_fix <- lin_bew |>
  transmute(cluster = linie, d_z_linie = d_z, d_b_linie = d_beitrag) |>
  inner_join(zerlegt |> transmute(cluster, d_z_fix = mittel_2024 - mittel_2016,
                                  d_b_fix = d_beitrag), by = "cluster")

cat("\nGemeinsame Startcluster:", nrow(vgl_fix), "\n")
cat("Rangkorrelation der z-Bewegung, fix gegen Linie:",
    round(cor(vgl_fix$d_z_fix, vgl_fix$d_z_linie, method = "spearman"), 3), "\n")
cat("Rangkorrelation des Beitragszuwachses          :",
    round(cor(vgl_fix$d_b_fix, vgl_fix$d_b_linie, method = "spearman"), 3), "\n")
print(vgl_fix |> mutate(rangdiff = abs(rank(-d_b_fix) - rank(-d_b_linie))) |>
        arrange(desc(rangdiff)) |> slice_head(n = 10), n = 10, width = 200)

write_csv(vgl_fix, file.path(OUT_DIR, "ff3_stammbaum_vs_fix.csv"))
write_csv(lin_bew, file.path(OUT_DIR, "ff3_stammbaum_bewegung.csv"))

# Drei Kennzahlen des Vergleichs
cat("\nPaare insgesamt                 :", nrow(vgl_fix), "\n")
cat("davon gleiche Richtung          :",
    sum(sign(vgl_fix$d_z_fix) == sign(vgl_fix$d_z_linie), na.rm = TRUE), "\n")
cat("mittlere abs. Verschiebung fix  :", sprintf("%.3f", mean(abs(vgl_fix$d_z_fix))), "\n")
cat("mittlere abs. Verschiebung Linie:", sprintf("%.3f", mean(abs(vgl_fix$d_z_linie))), "\n")

Linien, 2016 bis 2024 durchgehend intakt: 56 von 95 
Summe der Linien-Beiträge 2016 : 0.06865 | 2024 : 0.21114 
Gesamtes natives between   2016 : 0.13609 | 2024 : 0.36034 
Die Luecke ist der Teil der Landschaft, den die ueberlebenden Linien nicht abdecken.

=== Groesste Bewegung in positive Richtung ===
# A tibble: 12 × 14
   linie cl_bis n_von n_bis   z_von z_bis   d_z  tau_z d_beitrag bei_position
   <dbl>  <dbl> <int> <int>   <dbl> <dbl> <dbl>  <dbl>     <dbl>        <dbl>
 1    30    136    39    53 -0.518  1.03  1.55  0.667   0.00207      0.00268 
 2    54     87    50    47 -0.322  1.12  1.45  0.0556  0.00273      0.00434 
 3    88    176   154    65  0.0246 1.24  1.22  0.389   0.00557      0.0145  
 4    27    181   111   388  0.177  1.28  1.11  0.778   0.0353       0.0264  
 5    28     66    72   148  0.389  1.20  0.816 0       0.0112       0.0103  
 6    83    114    50   101 -0.284  0.497 0.782 0.278   0.00106      0.000931
 7    26    181   218   388  0.516  1.28  0.767 0.6

## Linien über die Wortähnlichkeit

In [20]:
cnt_part <- part  |> filter(cluster != -1) |> distinct(jahr, cluster) |> count(jahr, name = "n_part")
cnt_ctf  <- top50 |> distinct(year, cluster) |> count(year, name = "n_ctf") |> rename(jahr = year)
cat("Native Clusterzahl je Jahr, Partition gegen c-TF-IDF:\n")
print(full_join(cnt_part, cnt_ctf, by = "jahr"))

# Politische Lage jedes nativen Clusters je Jahr, freilaufende Partition
nat_frei <- part |>
  filter(cluster != -1) |>
  inner_join(zr |> select(subreddit, jahr, z, pct), by = c("subreddit", "jahr")) |>
  group_by(jahr, cluster) |>
  summarise(n_c = n(), mean_z = mean(z), mean_pct = mean(pct), .groups = "drop")

# FILLER wirft Tokens raus, die in zu vielen Clustern eines Jahres vorkommen und
# deshalb keine Nachfolger unterscheiden können.
jaccard_kanten <- function(TOPN, FILLER) {
  kk <- top50 |> filter(rang <= TOPN) |>
    group_by(year) |> mutate(K = n_distinct(cluster)) |> ungroup()
  fw <- kk |> group_by(year, token) |>
    summarise(share = n_distinct(cluster) / first(K), .groups = "drop") |>
    filter(share > FILLER)
  sets  <- kk |> anti_join(fw, by = c("year", "token")) |> select(year, cluster, token)
  sizes <- sets |> count(year, cluster, name = "sz")
  paare <- tibble(y0 = head(JAHRE, -1), y1 = tail(JAHRE, -1))
  pmap_dfr(paare, function(y0, y1) {
    a <- sets |> filter(year == y0) |> select(cl_a = cluster, token)
    b <- sets |> filter(year == y1) |> select(cl_b = cluster, token)
    inner_join(a, b, by = "token", relationship = "many-to-many") |>
      count(cl_a, cl_b, name = "inter") |>
      inner_join(sizes |> filter(year == y0) |> select(cl_a = cluster, sa = sz), by = "cl_a") |>
      inner_join(sizes |> filter(year == y1) |> select(cl_b = cluster, sb = sz), by = "cl_b") |>
      transmute(y0, y1, cl_a, cl_b, jacc = inter / (sa + sb - inter))
  })
}

linien_ketten <- function(kanten, THETA) {
  best <- kanten |> filter(jacc >= THETA) |>
    group_by(y0, cl_a) |> slice_max(jacc, n = 1, with_ties = FALSE) |> ungroup()
  start   <- sort(unique(top50$cluster[top50$year == min(JAHRE)]))
  aktiv   <- tibble(linie = start, jahr = min(JAHRE), cl = start, jacc_schritt = NA_real_)
  verlauf <- aktiv
  for (j in head(JAHRE, -1)) {
    schritt <- aktiv |>
      inner_join(best |> filter(y0 == j) |> select(cl = cl_a, cl_next = cl_b, jacc), by = "cl")
    if (nrow(schritt) == 0) break
    aktiv   <- schritt |> transmute(linie, jahr = j + 1L, cl = cl_next, jacc_schritt = jacc)
    verlauf <- bind_rows(verlauf, aktiv)
  }
  verlauf
}

Native Clusterzahl je Jahr, Partition gegen c-TF-IDF:
# A tibble: 9 × 3
   jahr n_part n_ctf
  <dbl>  <int> <int>
1  2016     95    95
2  2017     95    95
3  2018    120   120
4  2019    143   143
5  2020    165   165
6  2021    177   177
7  2022    201   201
8  2023    212   212
9  2024    200   200


In [21]:
rho_fuer <- function(TOPN, FILLER, THETA) {
  v <- linien_ketten(jaccard_kanten(TOPN, FILLER), THETA)
  w <- v |> left_join(nat_frei, by = c("jahr", "cl" = "cluster")) |>
    filter(jahr %in% c(min(JAHRE), max(JAHRE))) |>
    select(linie, jahr, mean_z) |>
    pivot_wider(names_from = jahr, values_from = mean_z, names_prefix = "mz_")
  sp24 <- paste0("mz_", max(JAHRE)); sp16 <- paste0("mz_", min(JAHRE))
  vgl <- w |> filter(!is.na(.data[[sp24]])) |>
    mutate(d_z_linie = .data[[sp24]] - .data[[sp16]]) |>
    inner_join(zerlegt |> transmute(linie = cluster,
                                    d_z_fix = mittel_2024 - mittel_2016,
                                    d_b_fix = d_beitrag), by = "linie")
  tibble(TOPN, FILLER, THETA,
         n_start = length(unique(v$linie[v$jahr == min(JAHRE)])),
         n_2024  = v |> filter(jahr == max(JAHRE)) |> distinct(linie) |> nrow(),
         n_vgl   = nrow(vgl),
         rho_dz  = cor(vgl$d_z_linie, vgl$d_z_fix, method = "spearman"),
         rho_db  = cor(vgl$d_z_linie, vgl$d_b_fix, method = "spearman"))
}

sweep <- pmap_dfr(expand_grid(TOPN = c(20, 30), FILLER = 0.30, THETA = c(0.10, 0.15, 0.20)),
                  rho_fuer)
print(sweep, n = 30, width = 200)

# A tibble: 6 × 8
   TOPN FILLER THETA n_start n_2024 n_vgl rho_dz rho_db
  <dbl>  <dbl> <dbl>   <int>  <int> <int>  <dbl>  <dbl>
1    20    0.3  0.1       95     90    90  0.499 0.0634
2    20    0.3  0.15      95     82    82  0.519 0.0814
3    20    0.3  0.2       95     77    77  0.562 0.0765
4    30    0.3  0.1       95     89    89  0.595 0.0594
5    30    0.3  0.15      95     80    80  0.616 0.0471
6    30    0.3  0.2       95     78    78  0.606 0.0408


In [22]:
TOPN0   <- 30
FILLER0 <- 0.30
THETA0  <- 0.15

k0 <- jaccard_kanten(TOPN0, FILLER0)
v0 <- linien_ketten(k0, THETA0)

kant <- k0 |> filter(jacc >= THETA0)
cat(sprintf("THETA = %.2f: Nachfolger je Cluster %.2f, Vorgaenger je Cluster %.2f\n",
            THETA0, mean((kant |> count(y0, cl_a))$n), mean((kant |> count(y1, cl_b))$n)))

endpunkte <- v0 |>
  left_join(nat_frei, by = c("jahr", "cl" = "cluster")) |>
  filter(jahr %in% c(2016, 2024)) |>
  select(linie, jahr, cl, mean_z, n_c) |>
  pivot_wider(names_from = jahr, values_from = c(cl, mean_z, n_c)) |>
  filter(!is.na(mean_z_2024)) |>
  mutate(d_z = mean_z_2024 - mean_z_2016,
         woerter_2016 = map_chr(cl_2016, woerter, jahr = 2016, n = 6),
         woerter_2024 = map_chr(cl_2024, woerter, jahr = 2024, n = 6))

cat("\nLinien 2016 bis 2024 durchgehend:", nrow(endpunkte),
    "von", length(unique(v0$linie[v0$jahr == 2016])), "Startclustern\n")

vgl0 <- endpunkte |>
  inner_join(zerlegt |> transmute(linie = cluster,
                                  d_z_fix = mittel_2024 - mittel_2016,
                                  d_b_fix = d_beitrag), by = "linie")
cat("Rangkorrelation z-Bewegung Linie gegen fixe Partition:",
    round(cor(vgl0$d_z, vgl0$d_z_fix, method = "spearman"), 3), "\n")
cat("Rangkorrelation Linie-z gegen fixen Beitragszuwachs  :",
    round(cor(vgl0$d_z, vgl0$d_b_fix, method = "spearman"), 3), "\n")
cat("\nNative Linien nach Richtung: nach oben", sum(endpunkte$d_z > 0),
    "| nach unten", sum(endpunkte$d_z < 0),
    "| mittlere d_z gewichtet", round(weighted.mean(endpunkte$d_z, endpunkte$n_c_2016), 4), "\n")

cat("\n=== Groesste Bewegung nach rechts entlang der nativen Linie ===\n")
print(endpunkte |> arrange(desc(d_z)) |>
        select(linie, cl_2024, n_c_2016, n_c_2024, mean_z_2016, mean_z_2024, d_z, woerter_2024) |>
        slice_head(n = 12), n = 12, width = 200)
cat("\n=== Groesste Bewegung nach links ===\n")
print(endpunkte |> arrange(d_z) |>
        select(linie, cl_2024, mean_z_2016, mean_z_2024, d_z, woerter_2024) |>
        slice_head(n = 8), n = 8, width = 200)

write_csv(endpunkte |> select(linie, cl_2016, cl_2024, n_c_2016, n_c_2024,
                              mean_z_2016, mean_z_2024, d_z, woerter_2016, woerter_2024),
          file.path(OUT_DIR, "ff3_linien_jaccard.csv"))

THETA = 0.15: Nachfolger je Cluster 4.71, Vorgaenger je Cluster 4.44

Linien 2016 bis 2024 durchgehend: 80 von 95 Startclustern
Rangkorrelation z-Bewegung Linie gegen fixe Partition: 0.616 
Rangkorrelation Linie-z gegen fixen Beitragszuwachs  : 0.047 

Native Linien nach Richtung: nach oben 31 | nach unten 49 | mittlere d_z gewichtet -0.4185 

=== Groesste Bewegung nach rechts entlang der nativen Linie ===
# A tibble: 12 × 8
   linie cl_2024 n_c_2016 n_c_2024 mean_z_2016 mean_z_2024   d_z
   <dbl>   <dbl>    <int>    <int>       <dbl>       <dbl> <dbl>
 1    30     136       39       53     -0.518        1.03  1.55 
 2    54      87       50       47     -0.322        1.12  1.45 
 3    88     176      154       65      0.0246       1.24  1.22 
 4    69      20       51      133      0.608        1.45  0.840
 5    28      66       72      148      0.389        1.20  0.816
 6    83     114       50      101     -0.284        0.497 0.782
 7    26     181      218      388      0.516      

In [23]:
fix_pct <- dat |>
  filter(jahr %in% c(2016, 2024)) |>
  group_by(cluster, jahr) |> summarise(mp = mean(pct), .groups = "drop") |>
  pivot_wider(names_from = jahr, values_from = mp, names_prefix = "mp_") |>
  mutate(d_pct_fix = mp_2024 - mp_2016)

rho_pct_fuer <- function(TOPN, FILLER, THETA) {
  v <- linien_ketten(jaccard_kanten(TOPN, FILLER), THETA)
  w <- v |> left_join(nat_frei, by = c("jahr", "cl" = "cluster")) |>
    filter(jahr %in% c(2016, 2024)) |>
    select(linie, jahr, mean_pct) |>
    pivot_wider(names_from = jahr, values_from = mean_pct, names_prefix = "mp_")
  vgl <- w |> filter(!is.na(mp_2024)) |>
    mutate(d_pct_linie = mp_2024 - mp_2016) |>
    inner_join(fix_pct |> select(linie = cluster, d_pct_fix), by = "linie")
  tibble(TOPN, FILLER, THETA, n_vgl = nrow(vgl),
         rho_pct = cor(vgl$d_pct_linie, vgl$d_pct_fix, method = "spearman"))
}

sweep_pct <- pmap_dfr(expand_grid(TOPN = c(20, 30), FILLER = 0.30, THETA = c(0.10, 0.15, 0.20)),
                      rho_pct_fuer)
cat("Rangbasierter Sweep (d_pct statt d_z):\n")
print(sweep_pct, n = 20, width = 160)

minj <- v0 |> group_by(linie) |>
  summarise(min_jacc = min(jacc_schritt, na.rm = TRUE), .groups = "drop")

ende <- v0 |>
  left_join(nat_frei, by = c("jahr", "cl" = "cluster")) |>
  filter(jahr %in% c(2016, 2024)) |>
  select(linie, jahr, cl, mean_z, mean_pct) |>
  pivot_wider(names_from = jahr, values_from = c(cl, mean_z, mean_pct)) |>
  filter(!is.na(mean_pct_2024)) |>
  left_join(minj, by = "linie") |>
  group_by(cl_2024) |> mutate(merge_n = n()) |> ungroup() |>
  mutate(d_z   = mean_z_2024 - mean_z_2016,
         d_pct = mean_pct_2024 - mean_pct_2016,
         woerter_2024 = map_chr(cl_2024, woerter, jahr = 2024, n = 6))

# anteil_alt: Anteil der 2016er Mitglieder des Startclusters, die 2024 im
# Endcluster der Linie sitzen. Anteil und nicht Jaccard, weil der Jaccard mit
# der wachsenden Population mechanisch fällt.
sub24 <- part |> filter(jahr == 2024, cluster != -1) |> select(subreddit, cl24 = cluster)

ueberlapp <- part |>
  filter(jahr == 2016, cluster != -1) |>
  select(subreddit, cl_2016 = cluster) |>
  inner_join(ende |> select(linie, cl_2016, cl_2024), by = "cl_2016") |>
  left_join(sub24, by = "subreddit") |>
  group_by(linie) |>
  summarise(n_alt       = n(),
            n_gemeinsam = sum(!is.na(cl24) & cl24 == cl_2024),
            anteil_alt  = n_gemeinsam / n_alt, .groups = "drop")

ende <- ende |> left_join(ueberlapp, by = "linie")

aa <- ende$anteil_alt
cat("\nGegenprobe Mitgliedschaft (nicht in die Kennzahlen eingerechnet):\n")
cat(sprintf("  Median %.2f | Q1 %.2f | Q3 %.2f\n", median(aa), quantile(aa, 0.25), quantile(aa, 0.75)))
cat(sprintf("  unter 0,05: %d von %d | unter 0,10: %d | unter 0,20: %d\n",
            sum(aa < 0.05), length(aa), sum(aa < 0.10), sum(aa < 0.20)))
cat(sprintf("  trotz min_jacc >= 0,25 unter 0,10: %d\n",
            sum(ende$anteil_alt < 0.10 & ende$min_jacc >= 0.25)))

cat("\nDie zehn schwaechsten Ketten (Fehlketten-Verdacht):\n")
print(ende |> arrange(anteil_alt) |>
        select(linie, cl_2024, anteil_alt, min_jacc, merge_n, d_pct, woerter_2024) |>
        slice_head(n = 10), n = 10, width = 200)
cat("Typisches Muster: generisches geteiltes Vokabular. Ortsangaben und NSFW-Woerter\n",
    "kommen in vielen Clustern gleichzeitig vor, der Jaccard kann zwischen ihnen nicht\n",
    "unterscheiden, und min_jacc faengt das nicht zuverlaessig ab.\n")

vgl_lin <- ende |>
  inner_join(fix_pct |> select(linie = cluster, d_pct_fix), by = "linie") |>
  inner_join(zerlegt |> transmute(linie = cluster, d_z_fix = mittel_2024 - mittel_2016),
             by = "linie")
stark <- vgl_lin |> filter(min_jacc >= 0.25)

cat(sprintf("\nLinien bis 2024: %d | min_jacc < 0.25: %d | in Merges: %d\n",
            nrow(vgl_lin), sum(vgl_lin$min_jacc < 0.25), sum(vgl_lin$merge_n > 1)))
cat(sprintf("  rangbasiert   d_pct: voll %.3f | starke Ketten (n = %d) %.3f\n",
            cor(vgl_lin$d_pct, vgl_lin$d_pct_fix, method = "spearman"),
            nrow(stark), cor(stark$d_pct, stark$d_pct_fix, method = "spearman")))
cat(sprintf("  niveaubasiert d_z  : voll %.3f | starke Ketten          %.3f\n",
            cor(vgl_lin$d_z, vgl_lin$d_z_fix, method = "spearman"),
            cor(stark$d_z, stark$d_z_fix, method = "spearman")))

cat("\n=== Staerkste Rechtsbewegung im Rang ===\n")
print(ende |> arrange(desc(d_pct)) |>
        select(linie, cl_2024, mean_pct_2016, mean_pct_2024, d_pct, min_jacc, merge_n, woerter_2024) |>
        slice_head(n = 12), n = 12, width = 200)
cat("\n=== Staerkste Linksbewegung im Rang ===\n")
print(ende |> arrange(d_pct) |>
        select(linie, cl_2024, mean_pct_2016, mean_pct_2024, d_pct, min_jacc, merge_n, woerter_2024) |>
        slice_head(n = 10), n = 10, width = 200)

write_csv(ende |> select(linie, cl_2016, cl_2024, mean_z_2016, mean_z_2024, d_z,
                         mean_pct_2016, mean_pct_2024, d_pct, min_jacc, merge_n,
                         n_alt, n_gemeinsam, anteil_alt, woerter_2024),
          file.path(OUT_DIR, "ff3_linien_jaccard_pct.csv"))

Rangbasierter Sweep (d_pct statt d_z):
# A tibble: 6 × 5
   TOPN FILLER THETA n_vgl rho_pct
  <dbl>  <dbl> <dbl> <int>   <dbl>
1    20    0.3  0.1     90   0.513
2    20    0.3  0.15    82   0.531
3    20    0.3  0.2     77   0.578
4    30    0.3  0.1     89   0.611
5    30    0.3  0.15    80   0.623
6    30    0.3  0.2     78   0.616


Warning message:
There were 5 warnings in `summarise()`.
The first warning was:
ℹ In argument: `min_jacc = min(jacc_schritt, na.rm = TRUE)`.
ℹ In group 10: `linie = 9`.
Caused by warning in `min()`:
! no non-missing arguments to min; returning Inf
ℹ Run ]8;;x-r-run:dplyr::last_dplyr_warnings()dplyr::last_dplyr_warnings()]8;; to see the 4 remaining warnings. 



Gegenprobe Mitgliedschaft (nicht in die Kennzahlen eingerechnet):
  Median 0.66 | Q1 0.32 | Q3 0.86
  unter 0,05: 12 von 80 | unter 0,10: 12 | unter 0,20: 13
  trotz min_jacc >= 0,25 unter 0,10: 5

Die zehn schwaechsten Ketten (Fehlketten-Verdacht):
# A tibble: 10 × 7
   linie cl_2024 anteil_alt min_jacc merge_n   d_pct
   <dbl>   <dbl>      <dbl>    <dbl>   <int>   <dbl>
 1     8     101    0          0.227       5 -0.489 
 2    34     101    0          0.189       5  0.0631
 3    44      82    0          0.217       2  0.0819
 4    45     101    0          0.227       5 -0.0502
 5    66      95    0          0.357       3 -0.194 
 6    69      20    0          0.326       2  0.188 
 7    77     184    0          0.2         3 -0.119 
 8    91     191    0          0.182       3  0.0987
 9    93     101    0          0.227       5 -0.254 
10     3      10    0.00120    0.3         1 -0.471 
   woerter_2024                              
   <chr>                                     
 1

In [24]:
linien_eval <- function(FILLER) {
  v <- linien_ketten(jaccard_kanten(TOPN0, FILLER), THETA0)
  minj <- v |> filter(!is.na(jacc_schritt)) |> group_by(linie) |>
    summarise(min_jacc = min(jacc_schritt), .groups = "drop")
  v |>
    left_join(nat_frei, by = c("jahr", "cl" = "cluster")) |>
    filter(jahr %in% c(2016, 2024)) |>
    select(linie, jahr, mean_z, mean_pct) |>
    pivot_wider(names_from = jahr, values_from = c(mean_z, mean_pct)) |>
    filter(!is.na(mean_pct_2024)) |>
    inner_join(minj, by = "linie") |>
    mutate(d_z = mean_z_2024 - mean_z_2016, d_pct = mean_pct_2024 - mean_pct_2016) |>
    inner_join(fix_pct |> select(linie = cluster, d_pct_fix), by = "linie") |>
    inner_join(zerlegt |> transmute(linie = cluster, d_z_fix = mittel_2024 - mittel_2016),
               by = "linie")
}

sweep_df <- map_dfr(c(0.15, 0.20, 0.25, 0.30, 0.40, 0.50), function(fl) {
  ev <- linien_eval(fl)
  map_dfr(c(0.00, 0.20, 0.25, 0.30), function(mc) {
    s <- ev |> filter(min_jacc >= mc)
    tibble(FILLER = fl, min_cut = mc, n = nrow(s),
           rho_pct = if (nrow(s) > 2) cor(s$d_pct, s$d_pct_fix, method = "spearman") else NA_real_,
           rho_dz  = if (nrow(s) > 2) cor(s$d_z,  s$d_z_fix,  method = "spearman") else NA_real_)
  })
})

zus <- sweep_df |> filter(!is.na(rho_pct))
cat(sprintf("Spanne ueber alle %d Kombinationen: rho_pct %.2f bis %.2f | rho_dz %.2f bis %.2f\n",
            nrow(zus), min(zus$rho_pct), max(zus$rho_pct), min(zus$rho_dz), max(zus$rho_dz)))
print(zus |> group_by(FILLER) |>
        summarise(n_komb = n(), rho_pct = mean(rho_pct), rho_dz = mean(rho_dz), .groups = "drop"))
hoechst <- zus |> slice_max(rho_pct, n = 1)
cat(sprintf("Hoechster Einzelwert: FILLER %.2f, min_cut %.2f, rho_pct %.3f\n",
            hoechst$FILLER, hoechst$min_cut, hoechst$rho_pct))

print(sweep_df, n = 40, width = 160)
cat("\nrho_pct als Matrix (Zeilen FILLER, Spalten min_cut):\n")
print(sweep_df |> select(FILLER, min_cut, rho_pct) |>
        pivot_wider(names_from = min_cut, values_from = rho_pct, names_prefix = "mc_"), width = 160)
cat("\nSteigt rho ueber die Spalten stark, aber ueber die Zeilen kaum, sitzt das Problem\n",
    "in einzelnen schwachen Ketten und nicht im Fuellwortfilter.\n")

write_csv(sweep_df, file.path(OUT_DIR, "ff3_sweep_filler_minjacc.csv"))

Spanne ueber alle 24 Kombinationen: rho_pct 0.57 bis 0.77 | rho_dz 0.54 bis 0.77
# A tibble: 6 × 4
  FILLER n_komb rho_pct rho_dz
   <dbl>  <int>   <dbl>  <dbl>
1   0.15      4   0.623  0.625
2   0.2       4   0.650  0.654
3   0.25      4   0.628  0.613
4   0.3       4   0.678  0.676
5   0.4       4   0.687  0.687
6   0.5       4   0.641  0.639
Hoechster Einzelwert: FILLER 0.30, min_cut 0.30, rho_pct 0.766
# A tibble: 24 × 5
   FILLER min_cut     n rho_pct rho_dz
    <dbl>   <dbl> <int>   <dbl>  <dbl>
 1   0.15    0       81   0.597  0.587
 2   0.15    0.2     75   0.598  0.591
 3   0.15    0.25    67   0.654  0.665
 4   0.15    0.3     60   0.644  0.658
 5   0.2     0       80   0.591  0.594
 6   0.2     0.2     75   0.672  0.676
 7   0.2     0.25    71   0.653  0.657
 8   0.2     0.3     58   0.684  0.688
 9   0.25    0       81   0.580  0.558
10   0.25    0.2     78   0.567  0.544
11   0.25    0.25    71   0.654  0.645
12   0.25    0.3     62   0.712  0.705
13   0.3     0       80  

## Vignetten, Labeln, Anhang

In [25]:
nachfolger <- read_csv(file.path(OUT_DIR, "ff3_linien_jaccard.csv"), show_col_types = FALSE) |>
  transmute(cluster       = as.integer(cl_2016),
            nachfolger_cl = as.integer(cl_2024),
            woerter_nachfolger_2024 = woerter_2024)
cat("Nachfolger:", nrow(nachfolger), "von", nrow(zerlegt),
    "fixen Clustern haben eine durchgehende Linie bis 2024\n")

N_JE_GRUPPE <- 10
gross <- beitrag_voll |> filter(n_subs >= 30)

kand <- bind_rows(
  beitrag_voll |> arrange(desc(d_beitrag))    |> slice_head(n = N_JE_GRUPPE) |> mutate(grund = "beitrag_hoch"),
  beitrag_voll |> arrange(d_beitrag)          |> slice_head(n = N_JE_GRUPPE) |> mutate(grund = "beitrag_runter"),
  beitrag_voll |> arrange(desc(anteil_24_pz)) |> slice_head(n = N_JE_GRUPPE) |> mutate(grund = "traegt_2024"),
  beitrag_voll |> filter(treiber == "Anteil", d_beitrag > 0) |> arrange(desc(d_beitrag)) |>
    slice_head(n = 5) |> mutate(grund = "nur_groesser"),
  gross |> arrange(desc(d_ext)) |> slice_head(n = N_JE_GRUPPE) |> mutate(grund = "raus_aus_mitte"),
  gross |> arrange(d_ext)       |> slice_head(n = N_JE_GRUPPE) |> mutate(grund = "rein_in_mitte")
) |>
  group_by(cluster) |>
  summarise(gruende = paste(sort(unique(grund)), collapse = "+"),
            across(c(n_subs, n_c_2016, n_c_2024, mittel_2016, mittel_2024,
                     anteil_16_pz, anteil_24_pz, d_beitrag, bei_position, bei_anteil,
                     treiber, pct_2016, pct_2024, d_pct, tau_pct, d_ext, tau_ext), first),
            .groups = "drop") |>
  mutate(woerter_2016 = map_chr(cluster, woerter, jahr = 2016, n = 12),
         mitglieder   = map_chr(cluster, ~ fix2016 |> filter(cluster == .x) |>
                                  pull(subreddit) |> head(10) |> paste(collapse = ", ")),
         label = "", kategorie = "") |>
  left_join(nachfolger, by = "cluster") |>
  arrange(desc(d_beitrag))

write_csv(kand, file.path(OUT_DIR, "ff3_kandidaten_TOLABEL.csv"))
print(kand |> select(cluster, gruende, n_subs, d_beitrag, treiber, anteil_24_pz,
                     d_pct, woerter_2016), n = 40, width = 230)

Nachfolger: 80 von 95 fixen Clustern haben eine durchgehende Linie bis 2024
# A tibble: 33 × 8
   cluster gruende                                 n_subs d_beitrag treiber 
     <dbl> <chr>                                    <int>     <dbl> <chr>   
 1      26 beitrag_hoch+raus_aus_mitte+traegt_2024    218  0.0210   Position
 2      17 beitrag_hoch+raus_aus_mitte+traegt_2024     95  0.0102   Position
 3       3 beitrag_hoch+traegt_2024                  1662  0.00874  Position
 4      88 beitrag_hoch+raus_aus_mitte+traegt_2024    154  0.00817  Position
 5       8 beitrag_hoch+raus_aus_mitte+traegt_2024     62  0.00505  Position
 6      87 beitrag_hoch+raus_aus_mitte+traegt_2024     66  0.00323  Position
 7      14 beitrag_hoch                                85  0.00322  Position
 8      73 beitrag_hoch                                69  0.00320  Position
 9      63 beitrag_hoch                               320  0.00314  Position
10      42 beitrag_hoch+raus_aus_mitte+traegt_2024     38 

In [26]:
# zerlegt hat eine Spalte namens woerter. Steht map_chr(cluster, woerter, ...)
# innerhalb von transmute(), findet dplyr zuerst die Spalte statt der Funktion
# und purrr bricht ab. Der Alias umgeht das.
wf <- woerter

fix_lab <- zerlegt |>
  arrange(desc(abs(d_beitrag))) |> slice_head(n = 15) |>
  transmute(cluster, n_subs, d_beitrag = round(d_beitrag, 4),
            woerter_2016 = map_chr(cluster, wf, jahr = 2016, n = 10))
cat("=== Fixe Partition: Top-15 Beitragstraeger ===\n")
print(fix_lab, n = 15, width = 240)

rob <- read_csv(file.path(OUT_DIR, "ff3_linien_jaccard_pct.csv"), show_col_types = FALSE) |>
  filter(min_jacc >= 0.30, merge_n == 1)
nat_lab <- bind_rows(rob |> slice_max(d_pct, n = 10), rob |> slice_min(d_pct, n = 10)) |>
  distinct(linie, .keep_all = TRUE) |>
  arrange(desc(d_pct)) |>
  transmute(linie, cl_2016, cl_2024, d_pct = round(d_pct, 3),
            woerter_2016 = map_chr(cl_2016, wf, jahr = 2016, n = 10),
            woerter_2024 = map_chr(cl_2024, wf, jahr = 2024, n = 10))
cat("\n=== Native Linien: Top-Wanderer ===\n")
print(nat_lab, n = 40, width = 280)

write_csv(fix_lab |> mutate(label = ""), file.path(OUT_DIR, "ff3_fix_zu_labeln.csv"))
write_csv(nat_lab |> mutate(label = ""), file.path(OUT_DIR, "ff3_nativ_zu_labeln.csv"))

=== Fixe Partition: Top-15 Beitragstraeger ===
# A tibble: 15 × 4
   cluster n_subs d_beitrag
     <dbl>  <int>     <dbl>
 1      26    218    0.021 
 2      17     95    0.0102
 3       3   1662    0.0087
 4      88    154    0.0082
 5      60     52   -0.0075
 6      92   1150   -0.0071
 7       8     62    0.005 
 8      87     66    0.0032
 9      14     85    0.0032
10      73     69    0.0032
11      63    320    0.0031
12      42     38    0.003 
13      36    183    0.0029
14       4     74    0.0029
15      53     56    0.0027
   woerter_2016                                                                 
   <chr>                                                                        
 1 anime, manga, episode, series, show, episodes, arc, mal, character, characte…
 2 london, pay, year, any, work, tax, money, years, british, though             
 3 sex, girl, sexy, looking, kik, cock, hot, cum, body, women                   
 4 nintendo, mario, game, wii, games, smash, raving, 

In [27]:
zer <- read_csv(file.path(OUT_DIR, "ff3_beitrag_zerlegung.csv"), show_col_types = FALSE)

# Die _pct-Datei führt nur woerter_2024, die Liste von 2016 steht in der Basisdatei
lin <- read_csv(file.path(OUT_DIR, "ff3_linien_jaccard_pct.csv"), show_col_types = FALSE) |>
  left_join(read_csv(file.path(OUT_DIR, "ff3_linien_jaccard.csv"), show_col_types = FALSE) |>
              select(linie, woerter_2016), by = "linie")

# Deutsche Zahlformate erst beim Schreiben, typografisches Minus wie im Text
dez <- function(x, n = 2) {
  formatC(x, format = "f", digits = n, decimal.mark = ",", big.mark = "") |>
    sub(pattern = "^-", replacement = "−")
}

# An der Wortgrenze kürzen: ein abgeschnittenes Token sieht nach Fehler aus
erste_n <- function(x, n = 5) {
  map_chr(str_split(x, ",\\s*"), function(w) {
    w <- w[nzchar(w)]
    if (length(w) <= n) paste(w, collapse = ", ")
    else paste0(paste(head(w, n), collapse = ", "), " …")
  })
}

# Escapen, was Pandoc sonst als Markup liest
esc <- function(x) {
  x |> str_replace_all(fixed("|"), "\\|") |>
       str_replace_all(fixed("_"), "\\_") |>
       str_replace_all(fixed("*"), "\\*")
}

md_tab <- function(df, datei, titel) {
  df     <- df |> mutate(across(everything(), ~ esc(as.character(.x))))
  kopf   <- paste0("| ", paste(names(df), collapse = " | "), " |")
  trenn  <- paste0("|", paste(rep("---", ncol(df)), collapse = "|"), "|")
  zeilen <- apply(df, 1, function(r) paste0("| ", paste(r, collapse = " | "), " |"))
  writeLines(c(paste0("Table: ", titel), "", kopf, trenn, zeilen), datei, useBytes = TRUE)
  cat("->", datei, "|", nrow(df), "Zeilen\n")
}

anhang <- zer |>
  arrange(desc(d_beitrag)) |>
  transmute(Cluster  = cluster,
            `n 2016` = n_c_2016,
            `n 2024` = n_c_2024,
            `z 2016` = dez(mittel_2016),
            `z 2024` = dez(mittel_2024),
            Beitrag  = dez(d_beitrag, 4),
            `Kennzeichnende Wörter 2016` = woerter)

write_csv(anhang, file.path(OUT_DIR, "ff3_anhang_cluster.csv"))
md_tab(anhang, file.path(OUT_DIR, "ff3_anhang_cluster.md"),
       paste0("Alle ", nrow(anhang), " Cluster der fixen Partition, nach Beitrag zum ",
              "Anstieg der Between-Varianz geordnet. z ist der mittlere Partisan-Score ",
              "des Clusters, standardisiert an der Verteilung aller Subreddits des ",
              "Jahres 2016. {#tbl:ff3-anhang-cluster}"))

# Im Anhang bewusst alle Linien, auch die Fehlketten: Der Anhang zeigt, was das
# Verfahren produziert hat, die Qualitätsspalten stehen daneben.
anhang_lin <- lin |>
  arrange(desc(d_z)) |>
  transmute(Linie = linie,
            `Cluster 2016` = cl_2016,
            `Cluster 2024` = cl_2024,
            `Überlappung`  = dez(anteil_alt),
            `min. Jaccard` = dez(min_jacc),
            `Veränderung z` = dez(d_z),
            `Wörter 2016` = woerter_2016,
            `Wörter 2024` = woerter_2024)

md_tab(anhang_lin, file.path(OUT_DIR, "ff3_anhang_linien.md"),
       paste0("Wortlisten aller ", nrow(anhang_lin), " verketteten Bündel im Vergleich ",
              "der Jahre 2016 und 2024. Überlappung ist der Anteil der 2016er ",
              "Mitglieder des Startclusters, die 2024 im Endcluster sitzen, min. Jaccard ",
              "die schwächste Wortüberlappung entlang der Kette. Ketten mit einer ",
              "Überlappung unter 0,05 sind die im Text benannten Fehlketten. ",
              "{#tbl:ff3-anhang-linien}"))

-> Auswertung_CSV/ff3_anhang_cluster.md | 95 Zeilen
-> Auswertung_CSV/ff3_anhang_linien.md | 80 Zeilen


In [28]:
source("cluster_namen.R")

In [29]:
anstieg_gesamt <- sum(zer$d_beitrag)

traeger <- bind_rows(zer |> slice_max(d_beitrag, n = 5),
                     zer |> slice_min(d_beitrag, n = 3)) |>
  distinct(cluster, .keep_all = TRUE) |>
  left_join(namen, by = "cluster") |>
  mutate(thema = coalesce(thema, paste0("Cluster ", cluster))) |>
  transmute(Cluster  = cluster,
            `Bündel` = thema,
            `n 2016` = n_c_2016,
            `z 2016` = dez(mittel_2016),
            `z 2024` = dez(mittel_2024),
            `Anteil am Anstieg` = paste0(dez(100 * d_beitrag / anstieg_gesamt, 1), " %"),
            `Kennzeichnende Wörter 2016` = erste_n(woerter, 5))

md_tab(traeger, file.path(OUT_DIR, "ff3_tab1_traeger.md"),
       paste0("Die fünf größten Beiträge zum Anstieg der Between-Varianz und die drei ",
              "größten Rückgänge. z ist der mittlere Partisan-Score des Clusters, ",
              "standardisiert an der Verteilung aller Subreddits des Jahres 2016. ",
              "{#tbl:ff3-traeger}"))

seitentab <- read_csv(file.path(OUT_DIR, "ff3_seiten.csv"), show_col_types = FALSE) |>
  mutate(Seite = if_else(seite == "positiv", "rechts vom Feldmittel 2016",
                                             "links vom Feldmittel 2016")) |>
  arrange(desc(zuwachs)) |>
  transmute(Seite,
            Cluster = n_cluster,
            `Subreddits 2016` = subs_2016,
            Zuwachs = dez(zuwachs, 4),
            `davon Position` = dez(via_position, 4),
            `davon Gewicht`  = dez(via_anteil, 4),
            `Anteil am Anstieg` = paste0(dez(anteil_am_zuwachs_pz, 1), " %"))

md_tab(seitentab, file.path(OUT_DIR, "ff3_tab2_seiten.md"),
       paste0("Beitrag zum Anstieg der Between-Varianz nach der Lage des Clusters im ",
              "Ausgangsjahr 2016. Getrennt wird am Mittelwert des Feldes und nicht am ",
              "Nullpunkt der Skala. Cluster, die im Verlauf die Seite wechseln, bleiben ",
              "ihrer Ausgangsseite zugeordnet. {#tbl:ff3-seiten}"))

print(traeger, width = 200)
print(seitentab, width = 200)

-> Auswertung_CSV/ff3_tab1_traeger.md | 8 Zeilen
-> Auswertung_CSV/ff3_tab2_seiten.md | 2 Zeilen
# A tibble: 8 × 7
  Cluster Bündel                          `n 2016` `z 2016` `z 2024`
    <dbl> <chr>                              <dbl> <chr>    <chr>   
1      26 Anime und Manga                      218 0,52     1,08    
2      17 Vereinigtes Königreich                95 0,83     1,28    
3       3 NSFW                                1662 0,00     −0,29   
4      88 Nintendo                             154 0,02     0,72    
5       8 Texas                                 62 0,72     1,11    
6      60 Wissenschaftsfragen                   52 −1,49    −0,90   
7      92 Sammelcluster (heterogen)           1150 0,31     0,01    
8      24 Nebenverdienste und Schnäppchen       67 −0,54    0,15    
  `Anteil am Anstieg` `Kennzeichnende Wörter 2016`                     
  <chr>               <chr>                                            
1 26,5 %              anime, manga, episode, series